# Step 6 on Kaggle: Whisper large-v3 + Audar on the pipeline's segments

Input: the folder your pipeline produced (`segments.csv` + the `segments/` WAV files), uploaded as a Kaggle dataset.
Output: the same `segments.csv` with the ASR columns added (`transcript_whisper`, `second_model`, `transcript_second`, `cer`, `n_diff_words`, `status`, `final_transcript`), saved in `/kaggle/working/asr_out/`.

**Settings (right panel):** Internet ON, Accelerator GPU T4, and your segments dataset added under *Input*. Do NOT install NeMo here.

In [19]:
!pip install -q qwen-asr librosa soundfile jiwer pandas

In [20]:
# ===== CONFIG (edit only this cell) =====
SEGMENTS_CSV  = None     # None = auto-find segments.csv under /kaggle/input.
                         # If you attached several datasets, set it explicitly, e.g.
                         # "/kaggle/input/delhab-segments/delhab/segments.csv"
LIMIT         = None     # e.g. 20 to test on the first 20 segments; None = all

WHISPER_ID    = "openai/whisper-large-v3"
WHISPER_BATCH = 4        # lower to 2 or 1 on CUDA out-of-memory
WHISPER_BEAMS = 5

AUDAR_ID      = "audarai/Audar-ASR-V1-Turbo"
AUDAR_LANG    = "Arabic"
AUDAR_BATCH   = 8        # lower to 4 on CUDA out-of-memory

OUT_DIR       = "/kaggle/working/asr_out"
SR            = 16000

## 1) Find the segments and load the audio

In [21]:
import glob, os, numpy as np, pandas as pd, soundfile as sf, librosa
from pathlib import Path

if SEGMENTS_CSV is None:
    found = glob.glob("/kaggle/input/**/segments.csv", recursive=True)
    if not found:
        raise FileNotFoundError("No segments.csv under /kaggle/input - add your dataset in the Input panel.")
    if len(found) > 1:
        raise RuntimeError("Several segments.csv found, set SEGMENTS_CSV explicitly:\n" + "\n".join(found))
    SEGMENTS_CSV = found[0]
BASE = Path(SEGMENTS_CSV).parent          # wav_file paths in the csv are relative to this folder
df = pd.read_csv(SEGMENTS_CSV)
if LIMIT:
    df = df.head(LIMIT).copy()
print("segments.csv :", SEGMENTS_CSV)
print("segments     :", len(df))
print("columns      :", list(df.columns))

paths = [BASE / f for f in df["wav_file"]]
missing = [str(p) for p in paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} wav files missing, first: {missing[0]}\n"
                            "Upload the segments/ folder together with segments.csv.")

def load(p):
    x, sr = sf.read(str(p), dtype="float32", always_2d=False)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if sr != SR:
        x = librosa.resample(x, orig_sr=sr, target_sr=SR)
    return x

audio = [load(p) for p in paths]
durs = np.array([len(a) / SR for a in audio])
print(f"audio loaded : {durs.sum()/60:.1f} min total | segment length min {durs.min():.1f}s, max {durs.max():.1f}s")
assert durs.max() <= 30.5, "A segment is longer than 30 s - the models take at most 30 s per input."
rms = np.array([float(np.sqrt((a ** 2).mean())) for a in audio])

os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = f"{OUT_DIR}/segments.csv"

segments.csv : /kaggle/input/datasets/ahmedhshousha/testdata/segments.csv
segments     : 386
columns      : ['segment_id', 'wav_file', 'start', 'end', 'duration', 'source_region_label', 'confidence', 'cut_type', 'snr_db', 'clean_noisy', 'transcript_whisper', 'transcript_mms', 'final_transcript', 'status', 'snr_energy_db', 'snr_wada_db', 'snr_disagree', 'noisy_label', 'clean_noisy_auto', 'cer', 'n_diff_words']
audio loaded : 66.0 min total | segment length min 3.3s, max 15.1s


## 2) Whisper large-v3 (saves a checkpoint when done)

In [22]:
import gc, torch
from tqdm.auto import tqdm
from transformers import AutoProcessor, WhisperForConditionalGeneration

dt = torch.float16
w_proc = AutoProcessor.from_pretrained(WHISPER_ID)
w_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_ID, torch_dtype=dt).to("cuda").eval()
print("Loaded", WHISPER_ID, "on", next(w_model.parameters()).device)

whisper_txt = []
for i in tqdm(range(0, len(audio), WHISPER_BATCH)):
    batch = audio[i:i + WHISPER_BATCH]
    feats = w_proc(batch, sampling_rate=SR, return_tensors="pt").input_features.to("cuda", dtype=dt)
    with torch.inference_mode():
        ids = w_model.generate(feats, language="ar", task="transcribe",
                               num_beams=WHISPER_BEAMS, max_new_tokens=300)
    texts = w_proc.batch_decode(ids, skip_special_tokens=True)
    for k, t in enumerate(texts):
        whisper_txt.append("" if rms[i + k] < 1e-3 else t.strip())     # silent -> blank (avoids hallucination)

df = df.copy()
df["transcript_whisper"] = whisper_txt
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Checkpoint saved:", OUT_CSV)

del w_model, w_proc
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory in use: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loaded openai/whisper-large-v3 on cuda:0


  0%|          | 0/97 [00:00<?, ?it/s]

Checkpoint saved: /kaggle/working/asr_out/segments.csv
GPU memory in use: 0.0 GB


## 3) Audar-ASR-V1-Turbo

In [23]:
from qwen_asr import Qwen3ASRModel

major, _ = torch.cuda.get_device_capability(0)
adt = torch.bfloat16 if major >= 8 else torch.float16        # T4 has no bf16
a_model = Qwen3ASRModel.from_pretrained(AUDAR_ID, dtype=adt, device_map="cuda:0",
                                        max_inference_batch_size=AUDAR_BATCH, max_new_tokens=256)
a_model.model.to(adt)
print("Loaded", AUDAR_ID, "| dtype", adt)

audar_txt = []
for i in tqdm(range(0, len(audio), AUDAR_BATCH)):
    batch = [(a, SR) for a in audio[i:i + AUDAR_BATCH]]
    out = a_model.transcribe(audio=batch, language=[AUDAR_LANG] * len(batch))
    audar_txt.extend(r.text.strip() for r in out)

df["second_model"] = "audar"
df["transcript_second"] = [("" if r < 1e-3 else t) for t, r in zip(audar_txt, rms)]
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Checkpoint saved:", OUT_CSV)

del a_model
gc.collect(); torch.cuda.empty_cache()

Loaded audarai/Audar-ASR-V1-Turbo | dtype torch.float16


  0%|          | 0/49 [00:00<?, ?it/s]

Checkpoint saved: /kaggle/working/asr_out/segments.csv


## 4) Compare and set the status
Same rule as `asr.py`: `accepted` if CER <= 0.02, `needs_review` if at most 3 words differ, otherwise `difficult`.

In [24]:
import re, jiwer

_DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
_ALEF = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا"})
def normalize(t):
    t = "" if t is None or (isinstance(t, float) and t != t) else str(t)
    t = _DIAC.sub("", t).replace("ـ", "").translate(_ALEF)
    t = t.replace("ة", "ه").replace("ى", "ي")
    t = re.sub(r"[^\w\s]", " ", t).replace("_", " ")
    return " ".join(t.split())

def decide(a, b):
    a, b = normalize(a), normalize(b)
    if not a or not b:
        return 1.0, 99, "difficult"
    w = jiwer.process_words(a, b)
    n_diff = w.substitutions + w.deletions + w.insertions
    cer = jiwer.cer(a, b)
    if cer <= 0.02:
        return cer, n_diff, "accepted"
    return (cer, n_diff, "needs_review") if n_diff <= 3 else (cer, n_diff, "difficult")

res = [decide(a, b) for a, b in zip(df["transcript_whisper"], df["transcript_second"])]
df["cer"] = [r[0] for r in res]
df["n_diff_words"] = [r[1] for r in res]
df["status"] = [r[2] for r in res]
df["final_transcript"] = [w if s == "accepted" else None for w, s in zip(df["transcript_whisper"], df["status"])]
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print(df["status"].value_counts().to_dict())
print("Saved:", OUT_CSV, " <- download this and put it over data\\<video>\\segments.csv")
df[["wav_file", "transcript_whisper", "transcript_second", "cer", "status"]].head(10)

{'difficult': 226, 'needs_review': 108, 'accepted': 52}
Saved: /kaggle/working/asr_out/segments.csv  <- download this and put it over data\<video>\segments.csv


,wav_file,transcript_whisper,transcript_second,cer,status
0,segments/segment_0001.wav,عن النبي صلى الله عليه وسلم يأتي علينا الزمان ...,يعني النبي صلى الله عليه وسلم يأتي على الناس ز...,0.207317,difficult
1,segments/segment_0002.wav,ومن لم يأكل الربا أصابه وباره. إذا تشوف بعض الناس,ومن لم يأكل ربا أصابه وباره، لذلك شوف بعض النصوص,0.187500,difficult
2,segments/segment_0003.wav,بعض الناس تاجر وغني وما عنده مشكلة أبداً يلعب ...,بعض الناس تاجر وغني وما عنده مشكلة أبدا يلعب ي...,0.103774,difficult
3,segments/segment_0004.wav,قبل الرجل قل ليتني مكانه من كل ما يشوف من الفت...,قبل رجلك قولي تاني مكانه من كل يوم نشوف من الف...,0.131250,difficult
4,segments/segment_0005.wav,كل هذا ما الحرام الحمد لله وكفى وأصلي وأسلم عل...,كل هذا ما الحرام؟ الحمد لله وكفى وأصلي وأسلم ع...,0.000000,accepted
5,segments/segment_0006.wav,يقول الله تبارك وتعالى أَحَسِبَ النَّاسُ أَنْ ...,يقول الله تبارك وتعالى: [ألف لام ميم] أحسب الن...,0.166667,needs_review
6,segments/segment_0007.wav,وَلَقَدْ فَتَنَّ الَّذِينَ مِنْ قَبْلِهِمْ فَل...,ولقد فتن الذين من قبلهم فليعلمن الله الذين صدق...,0.022989,needs_review
7,segments/segment_0008.wav,فتن تموج كما يموج البحر من الناس من يتشربها قل...,فتن تمو ج كما يموج البحر من الناس من يتشربها ق...,0.012658,accepted
8,segments/segment_0009.wav,ومنهم من ينكرها فؤاده فيبقى راسخا على دينه ثاب...,ومنهم من ينكرها فؤاده فيبقى راسخا على دينه ثاب...,0.000000,accepted
9,segments/segment_0010.wav,وأنواعها وأسباب الثبات عند حصول الفتن,وأنواعها وأسباب الثبات عند حصول الفتن.,0.000000,accepted
